# Processamento em Lote (Batch) - fNIRS | Tarefa: Stroop
Extração da resposta hemodinâmica cortical para a amostra completa (n=54) durante a tarefa Stroop.
A pipeline identifica os blocos 'Congruente' (1) e 'Incongruente' (2), executa a correção TDDR e filtra o sinal. O Modelo Linear Geral (GLM) extrai os *Betas* para cada condição e calcula automaticamente o contraste principal (Incongruente > Congruente) para cada canal longo, exportando os resultados para um dataset final consolidado.

In [1]:
import os
import mne
import mne_nirs
import pandas as pd
import numpy as np
from mne.preprocessing.nirs import optical_density, beer_lambert_law, scalp_coupling_index, temporal_derivative_distribution_repair
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.statistics import run_glm
import warnings

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# ==========================================
# 1. Configuração do Lote e Parâmetros
# ==========================================
BASE_DIR = r"C:\Users\guilh\Desktop\OneDrive - Universidade de Coimbra\PhD_Project_ARDITI_Guilherme Saldanha\Study2_Data Collection Files_PhD_Guilherme Saldanha_UC-UMa-ITI"
NUM_SUJEITOS = 54
TAREFA = "tk-stroop"

LIMIAR_SCI = 0.7
PPF = 0.1
LOW_CUTOFF_HZ = 0.01
HIGH_CUTOFF_HZ = 0.10
DISTANCIA_MIN_LONGA_M = 0.015

EVENTOS_STROOP = {
    "1": "Congruente",
    "2": "Incongruente"
}

dados_mestre_fnirs_stroop = []

print(f"A iniciar processamento neurofisiológico (fNIRS - STROOP) em lote para {NUM_SUJEITOS} sujeitos...\n")

# ==========================================
# 2. Ciclo de Processamento
# ==========================================
for i in range(1, NUM_SUJEITOS + 1):
    sujeito_id = f"sub-{i:03d}"
    pasta_nirs = os.path.join(BASE_DIR, sujeito_id, "nirs")
    
    print(f"[{sujeito_id}] A processar fNIRS...")
    
    try:
        if not os.path.exists(pasta_nirs):
            raise FileNotFoundError("Pasta 'nirs' não existe.")
            
        pastas_dentro_nirs = os.listdir(pasta_nirs)
        nome_pasta_run = next((p for p in pastas_dentro_nirs if TAREFA in p.lower() and os.path.isdir(os.path.join(pasta_nirs, p))), None)
        
        if not nome_pasta_run:
            raise FileNotFoundError(f"Pasta da tarefa '{TAREFA}' não encontrada.")
            
        pasta_alvo = os.path.join(pasta_nirs, nome_pasta_run)
        ficheiros_pasta = os.listdir(pasta_alvo)
        snirf_file = next((f for f in ficheiros_pasta if f.endswith('.snirf')), None)
        
        if not snirf_file:
            raise FileNotFoundError("Ficheiro .snirf em falta.")
            
        caminho_snirf = os.path.join(pasta_alvo, snirf_file)

        # -- Qualidade e Limpeza --
        raw_nirs = mne.io.read_raw_snirf(caminho_snirf, preload=True, verbose=False)
        raw_od = optical_density(raw_nirs)
        
        sci = scalp_coupling_index(raw_od)
        raw_od.info['bads'] = list(np.compress(sci < LIMIAR_SCI, raw_od.ch_names))

        raw_od_tddr = temporal_derivative_distribution_repair(raw_od)
        
        picks_validos = mne.pick_types(raw_od_tddr.info, fnirs=True, exclude='bads')
        distancias_od = mne.preprocessing.nirs.source_detector_distances(raw_od_tddr.info, picks=picks_validos)
        
        if np.sum(distancias_od <= 0.01) > 0:
            raw_od_ssr = mne_nirs.signal_enhancement.short_channel_regression(raw_od_tddr)
        else:
            raw_od_ssr = raw_od_tddr.copy()

        raw_haemo = beer_lambert_law(raw_od_ssr, ppf=PPF)
        raw_filtrado = raw_haemo.filter(LOW_CUTOFF_HZ, HIGH_CUTOFF_HZ, method="iir", verbose=False)

        # -- Isolamento Cortical e Triggers --
        picks_fnirs = mne.pick_types(raw_filtrado.info, fnirs=True, exclude="bads")
        distancias = mne.preprocessing.nirs.source_detector_distances(raw_filtrado.info, picks=picks_fnirs)
        picks_longos = picks_fnirs[distancias > DISTANCIA_MIN_LONGA_M]
        
        raw_cerebral = raw_filtrado.copy().pick(picks_longos)
        
        renomear = {cod: nome for cod, nome in EVENTOS_STROOP.items() if cod in raw_cerebral.annotations.description}
        raw_cerebral.annotations.rename(renomear)

        # -- GLM (Glover) --
        design_matrix = make_first_level_design_matrix(
            raw_cerebral,
            drift_model="cosine",
            high_pass=LOW_CUTOFF_HZ,
            hrf_model="glover"
        )
        
        glm_est = run_glm(raw_cerebral, design_matrix)
        colunas = design_matrix.columns.tolist()
        
        if "Incongruente" not in colunas or "Congruente" not in colunas:
            print(f"  [{sujeito_id}] AVISO: Marcadores incompletos na matriz. A saltar extração...")
            continue
            
        # -- Cálculo de Contrastes --
        # 1. Incongruente
        v_inc = np.zeros(len(colunas))
        v_inc[colunas.index("Incongruente")] = 1
        df_inc_hbo = glm_est.compute_contrast(v_inc).to_dataframe()
        df_inc_hbo = df_inc_hbo[df_inc_hbo['Chroma'] == 'hbo']
        
        # 2. Congruente
        v_con = np.zeros(len(colunas))
        v_con[colunas.index("Congruente")] = 1
        df_con_hbo = glm_est.compute_contrast(v_con).to_dataframe()
        df_con_hbo = df_con_hbo[df_con_hbo['Chroma'] == 'hbo']
        
        # 3. Contraste (Incongruente - Congruente)
        v_diff = np.zeros(len(colunas))
        v_diff[colunas.index("Incongruente")] = 1
        v_diff[colunas.index("Congruente")] = -1
        df_diff_hbo = glm_est.compute_contrast(v_diff).to_dataframe()
        df_diff_hbo = df_diff_hbo[df_diff_hbo['Chroma'] == 'hbo']

        # -- Compilação para Dataset --
        registo = {
            'ID': sujeito_id,
            'Tarefa': 'Stroop',
            'Canais_Rejeitados_SCI': len(raw_od.info['bads'])
        }
        
        for (_, row_inc), (_, row_con), (_, row_diff) in zip(df_inc_hbo.iterrows(), df_con_hbo.iterrows(), df_diff_hbo.iterrows()):
            canal = f"S{row_inc['Source']}_D{row_inc['Detector']}"
            
            # Betas Individuais
            registo[f'HbO_{canal}_Inc_Beta'] = row_inc['effect']
            registo[f'HbO_{canal}_Inc_pval'] = row_inc['p_value']
            registo[f'HbO_{canal}_Con_Beta'] = row_con['effect']
            registo[f'HbO_{canal}_Con_pval'] = row_con['p_value']
            
            # O Contraste Supremo (Incongruente > Congruente)
            registo[f'HbO_{canal}_Inc-Con_Beta'] = row_diff['effect']
            registo[f'HbO_{canal}_Inc-Con_pval'] = row_diff['p_value']
                
        dados_mestre_fnirs_stroop.append(registo)
        print(f"  [{sujeito_id}] GLM Stroop processado. Canais maus (SCI): {len(raw_od.info['bads'])}.")
        
    except Exception as e:
        print(f"  [{sujeito_id}] ERRO: {e}")

# ==========================================
# 3. Exportação da Base de Dados Consolidada
# ==========================================
if dados_mestre_fnirs_stroop:
    df_mestre = pd.DataFrame(dados_mestre_fnirs_stroop)
    pasta_exportacao = os.path.join(BASE_DIR, "derivatives", "mne-nirs")
    os.makedirs(pasta_exportacao, exist_ok=True)
    caminho_csv_final = os.path.join(pasta_exportacao, "dataset_mestre_fnirs_STROOP.csv")
    df_mestre.to_csv(caminho_csv_final, index=False)
    
    print(f"\nProcessamento fNIRS STROOP Concluído! Dados extraídos de {len(dados_mestre_fnirs_stroop)} sujeitos.")
    print(f"Base de dados exportada para:\n{caminho_csv_final}")
else:
    print("\nNenhum sujeito obteve dados válidos de fNIRS para exportação.")

A iniciar processamento neurofisiológico (fNIRS - STROOP) em lote para 54 sujeitos...

[sub-001] A processar fNIRS...
  [sub-001] GLM Stroop processado. Canais maus (SCI): 18.
[sub-002] A processar fNIRS...
  [sub-002] GLM Stroop processado. Canais maus (SCI): 16.
[sub-003] A processar fNIRS...
  [sub-003] GLM Stroop processado. Canais maus (SCI): 16.
[sub-004] A processar fNIRS...
  [sub-004] GLM Stroop processado. Canais maus (SCI): 16.
[sub-005] A processar fNIRS...
  [sub-005] GLM Stroop processado. Canais maus (SCI): 4.
[sub-006] A processar fNIRS...
  [sub-006] GLM Stroop processado. Canais maus (SCI): 16.
[sub-007] A processar fNIRS...
  [sub-007] GLM Stroop processado. Canais maus (SCI): 16.
[sub-008] A processar fNIRS...
  [sub-008] GLM Stroop processado. Canais maus (SCI): 16.
[sub-009] A processar fNIRS...
  [sub-009] GLM Stroop processado. Canais maus (SCI): 16.
[sub-010] A processar fNIRS...
  [sub-010] GLM Stroop processado. Canais maus (SCI): 16.
[sub-011] A processar fN